# 1. 네이버 뉴스 Crawling with Naver Open API

In [ ]:
# 네이버 검색 API 예제 -  블로그 검색 예제를 뉴스 검색 예제로 수정하여 실행
import os
import sys
import urllib.request
client_id = "fectRRiU4hHDSgWp0PYL"
client_secret = "CRcxk6VH6p"

encText = urllib.parse.quote("인공지능")

url = "https://openapi.naver.com/v1/search/blog?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)
    

In [ ]:
# 네이버 검색 API 예제 기반으로 네이버 검색 API 동작 방식 확인 (검색 결과 모두 받기)
# rescode가 200인 동안 start를 증가시켜서 계속 검색 (테스트를 위해 start는 25이하로 제한)
import os
import sys
import urllib.request
client_id = "fectRRiU4hHDSgWp0PYL"
client_secret = "CRcxk6VH6p"
encText = urllib.parse.quote("인공지능")
url = "https://openapi.naver.com/v1/search/blog?query=" + encText # JSON 결과

start = 1
display = 20
rescode = 200
while rescode == 200 and start < 100:
    new_url = url + f'&start={start}&display={display}'
    request = urllib.request.Request(new_url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read()
        print(response_body.decode('utf-8'))
    else:
        print("Error Code:" + rescode)

    start += display

# 2. NaverNewsCrawler 개발
1. 네이버 뉴스 API로 Crawling : **crawlNaverNews(keyword)** -> 파이썬 json 데이터
1. 응답데이터를 리스트에 저장 : **mergeResultToList(result_list, merged_list)** -> List
1. 응답데이터가 없을 때까지 반복
1. 리스트를 CSV 파일로 저장 : **saveJSONListToCSV(merged_list, csv_filename)**

In [32]:
# 검색 API 호출, 응답을 JSON 데이터로 return하는 함수 작성
# API 호출 결과에 문제가 있을 경우 exception으로 처리
# exception 발생 시 exception과 호출 url 확인 (try : ... except Exception as e: ... )

import os
import sys
import urllib.request
import json
import time


def crawlNaverNews(keyword, start, display=10):

    client_id = "fectRRiU4hHDSgWp0PYL"
    client_secret = "CRcxk6VH6p"
    encText = urllib.parse.quote("인공지능")
    url = "https://openapi.naver.com/v1/search/blog?query=" + encText # JSON 결과


    new_url = url + f'&start={start}&display={display}'
    request = urllib.request.Request(new_url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()

    if rescode==200:
        response_body = response.read()
        json_data = response_body.decode('utf-8')
        # json 문자열을 파이썬 객체로 변환
        py_data = json.loads(json_data)
        time.sleep(1)
        return py_data
    else:
        print("Error Code:" + rescode)
        return None
    

In [33]:
# 응답데이터를 리스트에 통합

def mergeResultToList(result_list, merged_list):
    
    for result in result_list:
        merged_list.append(result)


    

In [ ]:
# JSON의 list를 dataframe으로 변환하여 csv 파일로 저장



In [34]:
keyword = '날씨'

# API 호출 결과 전송된 JSON이 없거나, 전송된 JSON에 검색 결과가 없을 때까지
# 검색 결과를 JSON의 list로 저장 후 검색 API 추가 호출

# 검색 결과를 저장할 list 초기화
resultAll = []

start = 1

while start < 20:

    crawled_data = crawlNaverNews(keyword, start)

    if crawled_data:

        #print(crawled_data)
        print('crwaling 성공 :', start)
        resultAll += crawled_data['items']
        start += 10
    else:

        print('crawling 실패 :', start)
        break


crwaling 성공 : 1
crwaling 성공 : 11


In [36]:
resultAll[:2]

[{'title': '컴퓨터학원 실용적인 <b>인공지능</b> 수업과정',
  'link': 'https://blog.naver.com/foryou1035/224160610941',
  'description': '요즘 <b>인공지능</b>이 화두인데요. 처음에는 똑똑한 기능에 신기함이 컸지만, 시간이 갈수록 미묘한 감정이... 현재는 직장에서 <b>인공지능</b> 교육을 담당하게 되었고, 퇴근 후에는 취미로 웹사이트를 만들고 있어요.... ',
  'bloggername': 'Ssong Story ♡',
  'bloggerlink': 'blog.naver.com/foryou1035',
  'postdate': '20260126'},
 {'title': '<b>인공지능</b> 자격증 취득과정 및 취업 정보 정리',
  'link': 'https://blog.naver.com/gustlr00/224087574160',
  'description': '<b>인공지능</b> 자격증 취득과정 및 취업 정보 정리 “AI를 이해하는 사람과 그렇지 못한 사람의 차이가 점점 더 커지고 있다&quot;라는 말을 실감하게 되는 요즘, 직장 내에서도, 취업 시장에서도 <b>인공지능</b> 기술을 다룰... ',
  'bloggername': '〃다시 만나는 그날 까지 〃',
  'bloggerlink': 'blog.naver.com/gustlr00',
  'postdate': '20251125'}]

In [37]:
import os

# 1. 저장할 폴더 경로 설정
folder_path = './data'

# 2. 폴더가 없다면 생성
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"'{folder_path}' 폴더를 생성했습니다.")

# 3. 기존 코드 실행
data_df.to_csv(result_filename)

In [ ]:
import pandas as pd
data_df = pd.DataFrame(resultAll)
result_filename = f'./data/{keyword}_naver_news.csv'
data_df.to_csv(result_filename, encoding='utf-8-sig', index=False)

In [39]:
# 검색어 입력
keyword = '날씨'



# 검색 결과를 저장할 list 초기화
resultAll = []
merged_list = []
# 첫 검색 API 호출 데이터 초기화
start = 1
display = 10
while True:
    crawled_data = crawlNaverNews(keyword, start)
    # API 호출 성공 여부 출력
    if crawled_data:
        print(f'{keyword} [{start}] : 검색 성공')
    else:
        print(f'{keyword} [{start}] : 검색 실패')
        break
    
    # crawling된 데이터가 없으면 중지

    # 응답데이터 정리하여 리스트로 통합
    mergeResultToList(crawled_data, merged_list) 
    # 다음 검색 API 호출을 위한 파라미터 조정
    start += display


# 리스트를 csv 파일로 저장


날씨 [1] : 검색 성공
날씨 [11] : 검색 성공
날씨 [21] : 검색 성공
날씨 [31] : 검색 성공
날씨 [41] : 검색 성공
날씨 [51] : 검색 성공
날씨 [61] : 검색 성공
날씨 [71] : 검색 성공
날씨 [81] : 검색 성공
날씨 [91] : 검색 성공
날씨 [101] : 검색 성공
날씨 [111] : 검색 성공
날씨 [121] : 검색 성공
날씨 [131] : 검색 성공
날씨 [141] : 검색 성공
날씨 [151] : 검색 성공
날씨 [161] : 검색 성공
날씨 [171] : 검색 성공
날씨 [181] : 검색 성공
날씨 [191] : 검색 성공
날씨 [201] : 검색 성공
날씨 [211] : 검색 성공
날씨 [221] : 검색 성공
날씨 [231] : 검색 성공
날씨 [241] : 검색 성공
날씨 [251] : 검색 성공
날씨 [261] : 검색 성공
날씨 [271] : 검색 성공
날씨 [281] : 검색 성공
날씨 [291] : 검색 성공
날씨 [301] : 검색 성공
날씨 [311] : 검색 성공
날씨 [321] : 검색 성공
날씨 [331] : 검색 성공
날씨 [341] : 검색 성공
날씨 [351] : 검색 성공
날씨 [361] : 검색 성공
날씨 [371] : 검색 성공
날씨 [381] : 검색 성공
날씨 [391] : 검색 성공
날씨 [401] : 검색 성공
날씨 [411] : 검색 성공
날씨 [421] : 검색 성공
날씨 [431] : 검색 성공
날씨 [441] : 검색 성공
날씨 [451] : 검색 성공
날씨 [461] : 검색 성공
날씨 [471] : 검색 성공
날씨 [481] : 검색 성공
날씨 [491] : 검색 성공
날씨 [501] : 검색 성공
날씨 [511] : 검색 성공
날씨 [521] : 검색 성공
날씨 [531] : 검색 성공
날씨 [541] : 검색 성공
날씨 [551] : 검색 성공
날씨 [561] : 검색 성공
날씨 [571] : 검색 성공
날씨 [581] : 검색 성공
날씨 [591]

HTTPError: HTTP Error 400: Bad Request

In [31]:
# 저장한 파일을 dataframe으로 loading 해보기
loaded_data_df = pd.read_csv(result_filename, index_col=0)
loaded_data_df

""
